In [1]:
#DSC 550 Assignment 2
#Bilal Kudaimi 
#March 22, 2021

#Importing the necessary libraries
import json
import nltk
from nltk import pos_tag
from nltk import word_tokenize
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import sys
import unicodedata

In [2]:
#Importing the JSON file into a Pandas data frame
text_df = pd.read_json('controversial-comments.jsonl', lines = True)

#Since this dataframe is very large, I will randomly select 50000 rows from it to save programming time
random_50k_df = text_df.sample(n = 50000)
random_50k_df

,con,txt
360622,0,So what is the solution because while I agree ...
436464,0,"I think even if she won those swing states, Tr..."
546840,0,https://media.giphy.com/media/S3Ot3hZ5bcy8o/gi...
572172,0,Jill who?
717494,0,"Hahaha okay, the delusion is real. First Huma ..."
...,...,...
225987,1,Treason is the crime of betraying one's countr...
221572,0,Currently we give it to them for free...
315737,0,&gt; Better prove them right! That'll show 'em...
200876,1,"Lol go back to your safe space, reality won't ..."


In [3]:
#1A: Converting all text to lowercase letters
random_50k_df['txt'] = random_50k_df['txt'].str.lower()
random_50k_df['txt']

360622    so what is the solution because while i agree ...
436464    i think even if she won those swing states, tr...
546840    https://media.giphy.com/media/s3ot3hz5bcy8o/gi...
572172                                            jill who?
717494    hahaha okay, the delusion is real. first huma ...
                                ...                        
225987    treason is the crime of betraying one's countr...
221572             currently we give it to them for free...
315737    &gt; better prove them right! that'll show 'em...
200876    lol go back to your safe space, reality won't ...
845573    i got accused of sexism for not voting for hil...
Name: txt, Length: 50000, dtype: object

In [4]:
#1B: Removing all punctuation from the text

punctuation = dict.fromkeys(i for i in range(sys.maxunicode) if unicodedata.category(chr(i)).startswith('P'))

random_50k_df['txt'] = [string.translate(punctuation) for string in random_50k_df['txt']]
random_50k_df['txt']

360622    so what is the solution because while i agree ...
436464    i think even if she won those swing states tru...
546840         httpsmediagiphycommedias3ot3hz5bcy8ogiphygif
572172                                             jill who
717494    hahaha okay the delusion is real first huma is...
                                ...                        
225987    treason is the crime of betraying ones country...
221572                currently we give it to them for free
315737    gt better prove them right thatll show em\n\ny...
200876    lol go back to your safe space reality wont be...
845573    i got accused of sexism for not voting for hil...
Name: txt, Length: 50000, dtype: object

In [5]:
#1C: Removing all stop words

#Tokenizing the words first and appending each list of tokenized words into an empty list
tokenized_list = []
for string in random_50k_df['txt']:
    tokenized = word_tokenize(string)
    tokenized_list.append(tokenized)

#Removing the stop words by comparing to a list of known stop words
stop_words = stopwords.words('english')
random_50k_df['txt'] = [[word for word in sublist if word not in stop_words] for sublist in tokenized_list]
random_50k_df['txt']

360622    [solution, agree, capitalism, perfect, appears...
436464    [think, even, swing, states, trump, would, sti...
546840       [httpsmediagiphycommedias3ot3hz5bcy8ogiphygif]
572172                                               [jill]
717494    [hahaha, okay, delusion, real, first, huma, go...
                                ...                        
225987    [treason, crime, betraying, ones, country, mis...
221572                              [currently, give, free]
315737    [gt, better, prove, right, thatll, show, em, o...
200876    [lol, go, back, safe, space, reality, wont, kind]
845573    [got, accused, sexism, voting, hillary, disclo...
Name: txt, Length: 50000, dtype: object

In [6]:
#1D: Applying NLTK's PorterStemmer

porter = PorterStemmer()
random_50k_df['txt'] = [[porter.stem(word) for word in sublist] for sublist in random_50k_df['txt']]
random_50k_df['txt']

360622    [solut, agre, capit, perfect, appear, current,...
436464    [think, even, swing, state, trump, would, stil...
546840       [httpsmediagiphycommedias3ot3hz5bcy8ogiphygif]
572172                                               [jill]
717494    [hahaha, okay, delus, real, first, huma, go, p...
                                ...                        
225987    [treason, crime, betray, one, countri, mishand...
221572                                [current, give, free]
315737    [gt, better, prove, right, thatll, show, em, o...
200876    [lol, go, back, safe, space, realiti, wont, kind]
845573    [got, accus, sexism, vote, hillari, disclos, v...
Name: txt, Length: 50000, dtype: object

In [7]:
#2A: Converting each text entry into a word count vector

#Generating the word count vector for each text entry
#Each row is one text entry and each column is one unique word
count = CountVectorizer()
rejoined_words = [' '.join(i) for i in random_50k_df['txt']]
bag_of_words = count.fit_transform(rejoined_words)
word_count_vector = bag_of_words.toarray()

#Displaying the word count vector for each text entry
#This will be displayed as a sparse matrix, and thus, a majority of values will be zero
word_count_vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [9]:
#2B: Converting each text entry into a part-of-speech tag vector
#    This will be a list of list, with each list representing a POS tag vector for one text entry
[pos_tag(word_tokenize(string)) for string in rejoined_words]

[[('solut', 'NN'),
  ('agre', 'NN'),
  ('capit', 'NN'),
  ('perfect', 'VBP'),
  ('appear', 'JJ'),
  ('current', 'JJ'),
  ('success', 'NN')],
 [('think', 'VB'),
  ('even', 'RB'),
  ('swing', 'VBG'),
  ('state', 'NN'),
  ('trump', 'NN'),
  ('would', 'MD'),
  ('still', 'RB'),
  ('win', 'VB')],
 [('httpsmediagiphycommedias3ot3hz5bcy8ogiphygif', 'NN')],
 [('jill', 'NN')],
 [('hahaha', 'NN'),
  ('okay', 'MD'),
  ('delus', 'VB'),
  ('real', 'JJ'),
  ('first', 'JJ'),
  ('huma', 'NN'),
  ('go', 'VBP'),
  ('prison', 'NN'),
  ('hillari', 'NN')],
 [('poll', 'NN'), ('racist', 'NN'), ('bathroom', 'NN'), ('graffiti', 'NN')],
 [('actual', 'JJ'),
  ('good', 'JJ'),
  ('thing', 'NN'),
  ('accord', 'NN'),
  ('guy', 'NN'),
  ('trump', 'NN'),
  ('appoint', 'NN'),
  ('dismantl', 'NN'),
  ('epa', 'NN')],
 [('delet', 'NN')],
 [('delet', 'NN')],
 [('good', 'JJ')],
 [('gt', 'NN'),
  ('think', 'VBP'),
  ('agre', 'IN'),
  ('opposit', 'NN'),
  ('trump', 'JJ'),
  ('republican', 'JJ'),
  ('pretti', 'NN'),
  ('big', '

In [10]:
#2C: Converting each text entry into a term frequency-inverse document frequency (tfidf) vector

tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(rejoined_words)
# Show tf-idf feature matrix
TFIDFVector = feature_matrix.toarray()

#Displaying the TFIDFVector
#This will be displayed as a sparse matrix, and thus, a majority of values will be zero
TFIDFVector

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
'''
Follow up: Examples where each of the above three techniques would be useful

1. The word-count vector is useful when analyzing the word frequency of very large texts. For example, many 
   online word counters make use of word count vectors to function.

2. The POS tag vector is useful when one needs to know the parts of speech of words in large texts. For example,
   POS tagging allows one to search for all nouns, verbs, and adjectives within a text which can assist with NLP 
   tasks such as translation.

3. The TFIDF vector is useful for assigning weights to words, to help gauge which words are more important in a document.
   For example, TFIDF can help in sentiment analysis by identifying how much a word weighs into a document.
'''